# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`

This notebook provides a step-by-step template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata.to_json()

print("Dataset Title:")
print(metadata['name'])
print("\nDescription:")
print(metadata['description'])
print("\nPublication Date:")
print(metadata['datePublished'])
print("\nKeywords:")
print(metadata['keywords'])

## 2. Data Overview

Review available record sets and fields. All entities are referenced by their `@id` fields.

First, let's list the available record sets in the dataset and their corresponding fields and columns by `@id`.

In [ ]:
# Get all record sets defined in the metadata
record_sets = metadata.get('recordSet', [])
if not record_sets:
    print("No record sets found in the metadata.")
else:
    print("Record Sets available:")
    for rec_set in record_sets:
        if isinstance(rec_set, dict):
            print(f"- {rec_set.get('@id','[no id]')}")
        else:
            print(f"- {rec_set}")

    # For each record set, list its fields (by @id) if present
    print("\nFields and columns of each record set:")
    for rec_set in record_sets:
        rs_id = rec_set.get('@id') if isinstance(rec_set, dict) else rec_set
        rec_set_obj = next((x for x in metadata['recordSet'] if (isinstance(x, dict) and x.get('@id') == rs_id)), None)
        fields = []
        columns = []
        if rec_set_obj:
            fields = rec_set_obj.get('field', [])
            columns = rec_set_obj.get('column', [])
        print(f"\nRecord set: {rs_id}")
        if fields:
            print("  Fields (by @id):")
            for f in fields:
                print(f"    - {f.get('@id') if isinstance(f,dict) else f}")
        if columns:
            print("  Columns (by @id):")
            for c in columns:
                print(f"    - {c.get('@id') if isinstance(c,dict) else c}")


## 3. Data Extraction

Load data from each record set into a DataFrame for analysis.
All references use their `@id` values.

In [ ]:
# Collect all record set IDs
record_set_ids = []
for rec_set in metadata.get('recordSet', []):
    if isinstance(rec_set, dict):
        record_set_ids.append(rec_set.get('@id'))
    elif isinstance(rec_set, str):
        record_set_ids.append(rec_set)

# Preview records for each record set by @id and load DataFrames
dataframes = {}
for rec_id in record_set_ids:
    print(f"\nPreviewing records for record set: {rec_id}")
    records = list(dataset.records(record_set=rec_id))
    if records:
        print(records[0])  # Show first record as example
        df = pd.DataFrame(records)
        dataframes[rec_id] = df
        print(f"Loaded dataframe with shape: {df.shape} for {rec_id}")
    else:
        print(f"No records found for record set {rec_id}")

# Show columns for one sample record set
sample_rs_id = record_set_ids[0] if record_set_ids else None
if sample_rs_id and sample_rs_id in dataframes:
    print(f"\nColumns for record set {sample_rs_id}:")
    print(dataframes[sample_rs_id].columns.tolist())
    dataframes[sample_rs_id].head()


## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Refer to fields and columns using their `@id` values.

In [ ]:
# For demonstration, let's select a record set and a numeric field by their @id
# Suppose the main record set is available as the first ID
record_set_id = sample_rs_id
df = dataframes[record_set_id]

print(f"Available fields in this record set: {df.columns.tolist()}")

# Let's assume 'GAD_7_score' is a column (use its @id if available)
numeric_field_id = 'GAD_7_score'  # Replace with exact column @id from overview if different
group_field_id = 'village'  # Replace with exact column @id from overview if different

# Check if numeric_field_id exists
if numeric_field_id in df.columns:
    # Filter records with GAD_7_score > 10
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the GAD_7_score
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by village
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field_id}: Mean {numeric_field_id} per group")
        print(grouped_df.head())
else:
    print(f"Numeric field '{numeric_field_id}' not found in columns. Please check field names.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.
Use field and column `@id` values for reference.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram for GAD_7_score (or its @id column)
if numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

# Boxplot by group_field_id
if group_field_id in df.columns and numeric_field_id in df.columns:
    plt.figure(figsize=(10,4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

- This notebook demonstrates loading, exploring, and processing a Croissant-based dataset using `mlcroissant`, consistently referencing entities by their `@id` fields.
- The FAIR² Mental Health Survey Dataset provides valuable insight into mental health in Kilifi County, Kenya, with demographic and psychological scores for further analysis and AI readiness.
- The notebook can be extended to integrate additional processing, models, or visualizations as needed.